Fixed ALTER AGENT syntax to use MODIFY LIVE VERSION SET SPECIFICATION
*Co-authored with CoCo*

# Evaluating Cortex Agents: Hands-On Lab

**Duration:** 30-45 minutes  
**Scenario:** You're building a CMO-facing assistant that answers questions about campaign performance, budget allocation, and marketing strategy. The agent uses three tools: Cortex Analyst (structured data), Cortex Search (strategy documents), and an Agent Skill (executive summaries).

**What you'll do:**
1. Create the agent with all three tools
2. Build an evaluation dataset (12 questions across intent types)
3. Define metrics and run your first evaluation
4. Inspect results, identify weaknesses, and iterate
5. Version the improved agent and promote to production

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, tables, semantic view, Cortex Search service, and evaluation stage.

In [ ]:
# Connection setup — works in both Snowsight notebooks and local Jupyter
import os

try:
    # Snowsight notebook: session already exists
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    # Local Jupyter: create session from environment or connection config
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "CMO_EVAL_WH",
        "database": "CMO_EVAL_LAB",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()
    if os.environ.get("SNOWFLAKE_ACCOUNT") is None or os.environ.get("SNOWFLAKE_USER") is None or os.environ.get("SNOWFLAKE_PASSWORD") is None:
        raise ValueError(
            "Missing required environment variables. Please set SNOWFLAKE_ACCOUNT, SNOWFLAKE_USER, and SNOWFLAKE_PASSWORD."
        )

# Set context
session.sql("USE DATABASE CMO_EVAL_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE CMO_EVAL_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Database: {session.sql('SELECT CURRENT_DATABASE()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

---
## Section 1: Build the Agent

We'll create a Cortex Agent with three tools:
- **Cortex Analyst** — queries structured campaign performance data via a semantic view
- **Cortex Search** — retrieves marketing strategy documents via RAG
- **Agent Skill** — formats responses as executive summaries

In [ ]:
# Create the CMO Assistant agent
session.sql("""
CREATE OR REPLACE AGENT CMO_ASSISTANT
  COMMENT = 'Marketing/Finance assistant for campaign performance and strategy'
FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  response: |
    You are a CMO assistant that helps marketing leaders understand campaign performance,
    budget allocation, and strategic recommendations. Be concise and data-driven.
    When presenting financial data, always include the time period and round to 2 decimal places.
    When asked for a summary, brief, or executive-level view, format the response as:
    - A one-line headline insight
    - 3-5 bullet points with key findings
    - A recommended action
    Keep total length under 200 words.
  orchestration: |
    For quantitative questions about spend, revenue, ROI, conversions, or performance metrics, use the campaign_analytics tool.
    For questions about strategy, methodology, planning documents, or guidelines, use the strategy_search tool.
    For requests to summarize or create executive briefs, first gather data with the appropriate tool, then format as an executive summary.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: campaign_analytics
      description: "Query structured campaign performance data including spend, revenue, impressions, clicks, conversions, ROI, CPC, and CPA by channel and time period. Use for any quantitative marketing or financial question about campaign performance."
  - tool_spec:
      type: cortex_search
      name: strategy_search
      description: "Search marketing strategy documents, budget allocation methodology, attribution models, performance benchmarks, and planning briefs. Use for qualitative questions about strategy, process, methodology, or guidelines."
tool_resources:
  campaign_analytics:
    semantic_view: CMO_EVAL_LAB.PUBLIC.CMO_ANALYTICS
    execution_environment:
        type: warehouse
        warehouse: CMO_EVAL_WH
  strategy_search:
    name: CMO_EVAL_LAB.PUBLIC.STRATEGY_SEARCH_SVC
    max_results: "3"
$$
""").collect()

print("Agent CMO_ASSISTANT created successfully.")

In [ ]:
# Quick smoke test — verify the agent responds
result = session.sql("""
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT',
  $${
    "messages": [
      {
        "role": "user",
        "content": [
          {"type": "text", "text": "What was our total spend in 2024?"}
        ]
      }
    ]
  }$$
) AS response
""").collect()

import json
resp = json.loads(result[0]['RESPONSE'])

# Extract the assistant's text response
assistant_text = None
assistant_text = resp.get('content', [])[-2].get("text")
    

print("=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
if assistant_text:
    print(assistant_text)
else:
    print("(no text response found)")

print("\n" + "=" * 60)
print("FULL JSON")
print("=" * 60)
print(json.dumps(resp, indent=2))

---
## Section 2: Build the Evaluation Dataset

A good evaluation dataset covers the full intent distribution:
- **Happy paths** — common questions the agent should handle well
- **Multi-tool queries** — questions requiring coordination between tools
- **Edge cases** — specific, narrow queries
- **Out-of-scope** — questions the agent should refuse

We'll create 12 questions with ground truth that defines what a correct response looks like.

In [ ]:
# Create the evaluation questions table
session.sql("""
CREATE OR REPLACE TABLE EVAL_QUESTIONS (
    INPUT_QUERY   VARCHAR,
    GROUND_TRUTH  VARIANT
)
""").collect()

# Insert evaluation questions with ground truth
# intent: user's high-level goal (get_metric, compare_channels, summarize, retrieve_info, out_of_scope, etc.)
# process: complexity tier — single_tool, multi_tool, or refusal
questions = [
    # 1. Happy path — Analyst
    (
        "What was total spend across all channels in 2024?",
        '{"ground_truth_output": "Total marketing spend across all channels in 2024 was approximately $1,726,500. The response should present this as a single aggregate number covering all four channels (Paid Search, Social Media, Email, Display) for the full calendar year 2024. Values within ±1% are acceptable.", "intent": "get_metric", "process": "single_tool"}'
    ),
    # 2. Happy path — Analyst
    (
        "Which channel had the highest ROI in Q4 2024?",
        '{"ground_truth_output": "Email had the highest ROI in Q4 2024. Email ROI was approximately 11.3x (or ROAS of 12.3x), significantly outperforming other channels. The response should identify Email as the top performer and provide the ROI/ROAS figure. It should scope the answer explicitly to Q4 (October-December 2024).", "intent": "compare_channels", "process": "single_tool"}'
    ),
    # 3. Happy path — Search
    (
        "What is our attribution methodology?",
        '{"ground_truth_output": "The response should describe a data-driven multi-touch attribution model with a 30-day lookback window. Key details: first-touch weight 0.2, last-touch weight 0.3, middle touches share remaining 0.5 proportional to recency. It should mention that data refreshes daily with a 48-hour lag. The response should NOT fabricate details not in the strategy docs.", "intent": "retrieve_info", "process": "single_tool"}'
    ),
    # 4. Multi-tool — Analyst + Search
    (
        "Give me an executive summary of Q4 campaign performance for the board.",
        '{"ground_truth_output": "The response should combine quantitative Q4 data (total spend ~$590K, total revenue ~$2.06M, overall ROAS ~3.5x) with executive formatting: a headline insight, bullet points with key findings by channel, and a recommended action. It should be concise (under 200 words) and suitable for board-level communication.", "intent": "summarize", "process": "multi_tool"}'
    ),
    # 5. Multi-tool — Analyst + Search
    (
        "How does our actual Q4 spend allocation compare to what our channel strategy recommends?",
        '{"ground_truth_output": "The response should compare actual Q4 spend percentages by channel against the strategy document recommendations (Paid Search 40%, Social Media 25%, Email 10%, Display 15%, Flex 10%). It should calculate actual percentages from the spend data and note any significant deviations. This requires both querying spend data AND retrieving the channel strategy document.", "intent": "compare_actual_vs_plan", "process": "multi_tool"}'
    ),
    # 6. Edge case — Analyst (specific)
    (
        "What was our CPA for email in March 2024?",
        '{"ground_truth_output": "The CPA for Email in March 2024 was approximately $8.04 (calculated as $9,000 spend / 1,120 conversions). The response must be scoped to exactly March 2024 and the Email channel only. Values within ±$0.50 are acceptable.", "intent": "get_metric", "process": "single_tool"}'
    ),
    # 7. Happy path — Search
    (
        "What are our brand guidelines for reporting financial metrics?",
        '{"ground_truth_output": "The response should reference the brand guidelines document and include rules such as: currency in USD rounded to 2 decimal places, percentages to 1 decimal place, ROI expressed as a multiplier (e.g., 3.2x not 320%), time periods must be explicitly stated, and executive summaries must lead with the most impactful insight. It should NOT invent guidelines not in the documents.", "intent": "retrieve_info", "process": "single_tool"}'
    ),
    # 8. Complex — Analyst
    (
        "Compare paid search vs social media ROI trend over H2 2024 (July through December).",
        '{"ground_truth_output": "The response should show monthly or quarterly ROI for both Paid Search and Social Media during July-December 2024. Paid Search ROI should be significantly higher than Social Media throughout H2. Paid Search ROAS ranges roughly 3.3x-4.1x while Social Media ranges roughly 1.9x-2.1x. The trend should show both channels improving in Q4 due to holiday seasonality.", "intent": "compare_trends", "process": "single_tool"}'
    ),
    # 9. Out-of-scope — Refusal
    (
        "What's the weather forecast for tomorrow?",
        '{"ground_truth_output": "The response should clearly state that weather information is outside the agents capabilities and ideally redirect to what it can help with (marketing performance, campaign data, strategy questions). It should NOT fabricate a weather forecast or attempt to answer.", "intent": "out_of_scope", "process": "refusal"}'
    ),
    # 10. Multi-tool — Analyst + Search
    (
        "Create an executive brief on our highest-performing channel for 2024.",
        '{"ground_truth_output": "The response should identify Email as the highest-performing channel by ROI (approximately 10.5x ROAS for the year) and present the finding in executive summary format: headline insight, bullet points with supporting metrics (total Email spend ~$131.5K, revenue ~$1.46M, conversion rate ~7%), and a recommended action. Alternatively, Paid Search could be identified as highest by absolute revenue.", "intent": "summarize", "process": "multi_tool"}'
    ),
    # 11. Happy path — Analyst
    (
        "What campaigns did we run in Q1 2024?",
        '{"ground_truth_output": "The response should list the Q1 campaign name: Q1 Brand Awareness, which ran across all four channels (Paid Search, Social Media, Email, Display) during January-March 2024. It should scope the answer to Q1 only and not include campaigns from other quarters.", "intent": "list_items", "process": "single_tool"}'
    ),
    # 12. Out-of-scope — Refusal
    (
        "What did our competitors spend on marketing last year?",
        '{"ground_truth_output": "The response should state that competitive spend data is not available in the system. It should NOT fabricate competitor data. It may offer to help with the companys own marketing spend data as an alternative.", "intent": "out_of_scope", "process": "refusal"}'
    ),
]

# Insert all questions
for query, ground_truth in questions:
    escaped_query = query.replace("'", "''")
    session.sql(f"""
        INSERT INTO EVAL_QUESTIONS
        SELECT '{escaped_query}', PARSE_JSON('{ground_truth}')
    """).collect()

print(f"Inserted {len(questions)} evaluation questions.")
print("\nDistribution by process (complexity tier):")
session.sql("""
SELECT 
    GROUND_TRUTH:process::VARCHAR AS PROCESS,
    GROUND_TRUTH:intent::VARCHAR AS INTENT,
    COUNT(*) AS N
FROM EVAL_QUESTIONS
GROUP BY ALL
ORDER BY PROCESS, INTENT
""").show()

In [ ]:
# Drop existing dataset if it exists, then re-register
session.sql("DROP DATASET IF EXISTS CMO_EVAL_LAB.PUBLIC.CMO_EVAL_DATASET").collect()

# Register as an evaluation dataset
session.sql("""
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'CMO_EVAL_LAB.PUBLIC.EVAL_QUESTIONS',
  'CMO_EVAL_LAB.PUBLIC.CMO_EVAL_DATASET',
  OBJECT_CONSTRUCT(
    'query_text', 'INPUT_QUERY',
    'expected_tools', 'GROUND_TRUTH'
  )
)
""").collect()

print("Dataset registered: CMO_EVAL_DATASET")
session.sql("SHOW DATASETS IN SCHEMA CMO_EVAL_LAB.PUBLIC").show()

---
## Section 3: Define Metrics and Run the Evaluation

We'll use three metrics:
- **answer_correctness** (built-in) — compares agent output to ground truth
- **logical_consistency** (built-in, reference-free) — checks internal consistency of planning/execution
- **tool_selection** (custom) — evaluates whether the agent chose the right tool for each query

In [ ]:
# Write the evaluation config YAML
eval_config_yaml = """
# Cortex Agent Evaluation Configuration
# CMO Assistant — Baseline Run

evaluation:
  agent_params:
    agent_name: "CMO_ASSISTANT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "CMO Assistant evaluation"
  source_metadata:
    type: "dataset"
    dataset_name: "CMO_EVAL_DATASET"

metrics:
  # Built-in: compare agent output to ground truth
  - "answer_correctness"
  # Built-in: reference-free consistency check
  - "logical_consistency"
  # Custom: evaluate tool selection quality
  - name: "tool_selection"
    score_ranges:
      min_score: [1, 3]
      median_score: [4, 6]
      max_score: [7, 10]
    prompt: |
      Evaluate whether the agent selected the correct tool(s) for the user's query.

      User query: {{input}}
      Tools used: {{tool_info}}
      Expected behavior: {{ground_truth}}
      Agent response: {{output}}

      Rate from 1-10:
      1-3 = Wrong tool selected, unnecessary tool calls, or failed to use a tool when one was needed
      4-6 = Partially correct tool selection (used the right primary tool but missed a secondary tool, or made redundant calls)
      7-10 = Optimal tool selection for the query intent

      Evaluation criteria:
      - Did the agent use campaign_analytics for quantitative questions about spend, revenue, ROI, etc.?
      - Did it use strategy_search for qualitative questions about methodology, strategy, or guidelines?
      - Did it correctly combine tools for multi-faceted questions?
      - For out-of-scope questions, did it avoid calling tools and instead refuse gracefully?

      Provide your score as a single integer.
""".strip()

# Write YAML to a local temp file, then PUT to stage
import tempfile

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(eval_config_yaml)
    yaml_path = f.name

session.sql(f"""
PUT 'file://{yaml_path}' @CMO_EVAL_LAB.PUBLIC.EVAL_STAGE
  AUTO_COMPRESS=FALSE
  OVERWRITE=TRUE
""").collect()

print("Evaluation config uploaded to @EVAL_STAGE")
session.sql("LIST @CMO_EVAL_LAB.PUBLIC.EVAL_STAGE").show()

In [ ]:
# Start the baseline evaluation run
session.sql("""
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'baseline-v1'),
  '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
)
""").collect()

print("Evaluation 'baseline-v1' started. This will take 2-5 minutes...")

In [ ]:
# Poll evaluation status — re-run this cell until STATUS shows COMPLETED
import time

for i in range(20):
    status = session.sql("""
    CALL EXECUTE_AI_EVALUATION(
      'STATUS',
      OBJECT_CONSTRUCT('run_name', 'baseline-v1'),
      '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
    )
    """).collect()
    
    current_status = status[0]['STATUS'] if 'STATUS' in status[0].as_dict() else str(status[0])
    print(f"[{i+1}/20] Status: {current_status}")
    
    if 'COMPLETED' in str(current_status).upper():
        print("\nEvaluation complete!")
        break
    
    time.sleep(30)
else:
    print("\nStill running — re-run this cell to check again.")

---
## Section 4: Inspect Results and Iterate

Now we'll look at the evaluation results, identify the weakest areas, and make a targeted improvement to the agent's instructions.

In [ ]:
# View overall scores by metric
session.sql("""
SELECT
    METRIC_NAME,
    METRIC_TYPE,
    COUNT(*) AS NUM_RECORDS,
    ROUND(AVG(EVAL_AGG_SCORE), 3) AS AVG_SCORE,
    ROUND(MIN(EVAL_AGG_SCORE), 3) AS MIN_SCORE,
    ROUND(MAX(EVAL_AGG_SCORE), 3) AS MAX_SCORE
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
))
GROUP BY METRIC_NAME, METRIC_TYPE
ORDER BY AVG_SCORE ASC
""").show()

In [ ]:
# View per-question detail — sorted by lowest scores to find failures
session.sql("""
SELECT
    INPUT,
    METRIC_NAME,
    ROUND(EVAL_AGG_SCORE, 3) AS SCORE,
    METRIC_CALLS[0]:explanation::VARCHAR AS EXPLANATION
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
))
ORDER BY EVAL_AGG_SCORE ASC
LIMIT 10
""").show(max_width=120)

In [ ]:
# Deep dive by process (complexity tier) and intent
session.sql("""
SELECT
    e.GROUND_TRUTH:process::VARCHAR AS PROCESS,
    e.GROUND_TRUTH:intent::VARCHAR AS INTENT,
    r.INPUT,
    r.METRIC_NAME,
    ROUND(r.EVAL_AGG_SCORE, 3) AS SCORE,
    LEFT(r.OUTPUT, 200) AS AGENT_RESPONSE_PREVIEW
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
)) r
JOIN EVAL_QUESTIONS e
    ON r.INPUT = e.INPUT_QUERY
WHERE e.GROUND_TRUTH:process::VARCHAR IN ('multi_tool', 'refusal')
ORDER BY PROCESS, INTENT, METRIC_NAME
""").show(max_width=120)

In [ ]:
# Pivot table: average score by process x intent x metric
session.sql("""
SELECT
    e.GROUND_TRUTH:process::VARCHAR AS PROCESS,
    e.GROUND_TRUTH:intent::VARCHAR AS INTENT,
    ROUND(AVG(CASE WHEN r.METRIC_NAME = 'answer_correctness' THEN r.EVAL_AGG_SCORE END), 3) AS ANSWER_CORRECTNESS,
    ROUND(AVG(CASE WHEN r.METRIC_NAME = 'tool_selection' THEN r.EVAL_AGG_SCORE END), 3) AS TOOL_SELECTION,
    ROUND(AVG(CASE WHEN r.METRIC_NAME = 'logical_consistency' THEN r.EVAL_AGG_SCORE END), 3) AS LOGICAL_CONSISTENCY,
    COUNT(DISTINCT r.INPUT) AS N_QUESTIONS
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
)) r

JOIN EVAL_QUESTIONS e
    ON r.INPUT = e.INPUT_QUERY
GROUP BY ALL
""").show(max_width=150)

### Iterate: Improve the Agent

Based on the results above, we'll make a targeted fix. Common failure modes for this agent include:
- **Multi-tool coordination** — the agent may not combine analytics + search or analytics + summary tools effectively
- **Out-of-scope handling** — the agent may attempt to answer questions it should refuse

We'll update the orchestration instructions to be more explicit about these scenarios.

In [ ]:
# Improve the agent's instructions based on evaluation findings
session.sql("""
ALTER AGENT CMO_ASSISTANT MODIFY LIVE VERSION SET SPECIFICATION = $$
models:
  orchestration: auto
instructions:
  response: |
    You are a CMO assistant that helps marketing leaders understand campaign performance,
    budget allocation, and strategic recommendations. Be concise and data-driven.
    When presenting financial data, always include the time period and round to 2 decimal places.
    Express ROI as a multiplier (e.g., 3.2x).
  orchestration: |
    TOOL SELECTION RULES (follow strictly):
    1. QUANTITATIVE questions (spend, revenue, ROI, CPA, conversions, performance metrics) -> use campaign_analytics
    2. QUALITATIVE questions (strategy, methodology, guidelines, planning, benchmarks) -> use strategy_search
    3. COMPARISON questions that reference both data AND strategy documents -> use campaign_analytics FIRST, then strategy_search
    4. EXECUTIVE SUMMARY requests -> gather data with the appropriate tool first, then use executive_summary to format
    5. OUT-OF-SCOPE questions (weather, competitors, anything not about our marketing data or strategy) -> DO NOT call any tool. Politely decline and explain what you CAN help with.

    MULTI-TOOL COORDINATION:
    - When a question requires both quantitative data and qualitative context, ALWAYS call both tools.
    - Example: "How does our spend compare to strategy recommendations?" requires campaign_analytics for actual spend AND strategy_search for the recommended allocation.
    - For executive briefs, ALWAYS gather data before formatting with executive_summary.
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: campaign_analytics
      description: "Query structured campaign performance data including spend, revenue, impressions, clicks, conversions, ROI, CPC, and CPA by channel and time period. Use for any quantitative marketing or financial question about campaign performance."
  - tool_spec:
      type: cortex_search
      name: strategy_search
      description: "Search marketing strategy documents, budget allocation methodology, attribution models, performance benchmarks, and planning briefs. Use for qualitative questions about strategy, process, methodology, or guidelines."
tool_resources:
  campaign_analytics:
    semantic_view: CMO_EVAL_LAB.PUBLIC.CMO_ANALYTICS
    execution_environment:
        type: warehouse
        warehouse: CMO_EVAL_WH
  strategy_search:
    name: CMO_EVAL_LAB.PUBLIC.STRATEGY_SEARCH_SVC
    max_results: "3"
$$
""").collect()

print("Agent instructions updated with improved orchestration guidance.")

In [ ]:
# Commit the improved version as an immutable snapshot
session.sql("""
ALTER AGENT CMO_ASSISTANT COMMIT
  COMMENT = 'Improved multi-tool orchestration and explicit out-of-scope handling'
""").collect()

print("Version committed. Running improved evaluation...")

# Start the second evaluation run
session.sql("""
CALL EXECUTE_AI_EVALUATION(
  'START',
  OBJECT_CONSTRUCT('run_name', 'improved-v2'),
  '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
)
""").collect()

print("Evaluation 'improved-v2' started. This will take 2-5 minutes...")

In [ ]:
# Poll status for improved run
import time

for i in range(20):
    status = session.sql("""
    CALL EXECUTE_AI_EVALUATION(
      'STATUS',
      OBJECT_CONSTRUCT('run_name', 'improved-v2'),
      '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
    )
    """).collect()
    
    current_status = status[0]['STATUS'] if 'STATUS' in status[0].as_dict() else str(status[0])
    print(f"[{i+1}/20] Status: {current_status}")
    
    if 'COMPLETED' in str(current_status).upper():
        print("\nEvaluation complete!")
        break
    
    time.sleep(30)
else:
    print("\nStill running — re-run this cell to check again.")

In [ ]:
# Compare baseline vs improved — side by side
session.sql("""
WITH baseline AS (
    SELECT METRIC_NAME, ROUND(AVG(EVAL_AGG_SCORE), 3) AS BASELINE_AVG
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
    ))
    GROUP BY METRIC_NAME
),
improved AS (
    SELECT METRIC_NAME, ROUND(AVG(EVAL_AGG_SCORE), 3) AS IMPROVED_AVG
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'improved-v2'
    ))
    GROUP BY METRIC_NAME
)
SELECT
    b.METRIC_NAME,
    b.BASELINE_AVG,
    i.IMPROVED_AVG,
    ROUND(i.IMPROVED_AVG - b.BASELINE_AVG, 3) AS DELTA
FROM baseline b
JOIN improved i ON b.METRIC_NAME = i.METRIC_NAME
ORDER BY DELTA DESC
""").show()

In [ ]:
# Granular comparison grouped by intent and process
session.sql("""
WITH baseline AS (
    SELECT INPUT, METRIC_NAME, EVAL_AGG_SCORE AS BASELINE_SCORE
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'baseline-v1'
    ))
),
improved AS (
    SELECT INPUT, METRIC_NAME, EVAL_AGG_SCORE AS IMPROVED_SCORE
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'improved-v2'
    ))
)
SELECT
    e.GROUND_TRUTH:process::VARCHAR AS PROCESS,
    e.GROUND_TRUTH:intent::VARCHAR AS INTENT,
    b.METRIC_NAME,
    ROUND(AVG(b.BASELINE_SCORE), 3) AS AVG_BASELINE,
    ROUND(AVG(i.IMPROVED_SCORE), 3) AS AVG_IMPROVED,
    ROUND(AVG(i.IMPROVED_SCORE) - AVG(b.BASELINE_SCORE), 3) AS DELTA,
    COUNT(*) AS N
FROM baseline b
JOIN improved i
    ON b.INPUT = i.INPUT AND b.METRIC_NAME = i.METRIC_NAME
JOIN EVAL_QUESTIONS e
    ON b.INPUT = e.INPUT_QUERY
GROUP BY ALL
ORDER BY PROCESS, INTENT, b.METRIC_NAME
""").show(n=50, max_width=150)

---
## Section 5: Version and Promote

With the improved version showing better scores, we'll promote it to production using the alias system. This means application code that references the `production` alias will automatically pick up the improved version — no code changes required.

In [ ]:
# View all versions of the agent
session.sql("SHOW VERSIONS IN AGENT CMO_ASSISTANT").show()

In [ ]:
# Promote the improved version to production
# Recreate the live version if it was consumed by a prior commit
try:
    session.sql("ALTER AGENT CMO_ASSISTANT ADD LIVE VERSION FROM LAST").collect()
    print("Recreated live version from last committed version.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Live version already exists, proceeding.")
    else:
        raise

# Commit the live version to create an immutable named version
session.sql("ALTER AGENT CMO_ASSISTANT COMMIT COMMENT = 'Improved orchestration from eval findings'").collect()
print("Committed live version as a new named version.")

# The default version automatically points to the last committed version.
# Assign a 'production' alias to the latest committed version for explicit routing.
versions_df = session.sql("SHOW VERSIONS IN AGENT CMO_ASSISTANT").collect()
latest_version = versions_df[-1]['name']  # last row = most recent commit
session.sql(f"ALTER AGENT CMO_ASSISTANT MODIFY VERSION {latest_version} SET ALIAS = production").collect()
print(f"Assigned 'production' alias to {latest_version}.")

# Verify
session.sql("SHOW VERSIONS IN AGENT CMO_ASSISTANT").show()

In [ ]:
# Test the production version — DATA_AGENT_RUN routes to the default (latest committed) version
import json

result = session.sql("""
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT',
  $${
    "messages": [
      {
        "role": "user",
        "content": [
          {"type": "text", "text": "Which channel should we increase budget for next quarter based on 2024 performance?"}
        ]
      }
    ],
    "version": "production"
  }$$
) AS response
""").collect()

resp = json.loads(result[0]['RESPONSE'])
assistant_text = next((c.get('text') for c in resp.get('content', []) if c.get('type') == 'text'), None)

print("Production agent response:")
print(assistant_text[:600] if assistant_text else "(no text response found)")

---
## Section 6: Scheduled Regression Detection

In production, you need to catch regressions from external factors — model updates, data drift, tool configuration changes — not just regressions from your own code changes.

The pattern: a **Snowflake Task** runs `EXECUTE_AI_EVALUATION` on a schedule (daily or weekly), stores scores in a history table, and alerts you when scores drop below a threshold.

In [ ]:
# Create a table to store evaluation score history over time
session.sql("""
CREATE OR REPLACE TABLE EVAL_SCORE_HISTORY (
    RUN_NAME        VARCHAR,
    RUN_TIMESTAMP   TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP(),
    METRIC_NAME     VARCHAR,
    AVG_SCORE       FLOAT,
    MIN_SCORE       FLOAT,
    MAX_SCORE       FLOAT,
    NUM_RECORDS     INT
)
""").collect()

# Seed it with our two existing runs
for run_name in ['baseline-v1', 'improved-v2']:
    session.sql(f"""
    INSERT INTO EVAL_SCORE_HISTORY (RUN_NAME, METRIC_NAME, AVG_SCORE, MIN_SCORE, MAX_SCORE, NUM_RECORDS)
    SELECT
        '{run_name}',
        METRIC_NAME,
        ROUND(AVG(EVAL_AGG_SCORE), 4),
        ROUND(MIN(EVAL_AGG_SCORE), 4),
        ROUND(MAX(EVAL_AGG_SCORE), 4),
        COUNT(*)
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', '{run_name}'
    ))
    GROUP BY METRIC_NAME
    """).collect()

print("Score history seeded with baseline-v1 and improved-v2 results.")
session.sql("SELECT * FROM EVAL_SCORE_HISTORY ORDER BY RUN_TIMESTAMP, METRIC_NAME").show()

In [ ]:
# Create a stored procedure that runs an eval and checks for regression
# First ensure SYSADMIN has the required privilege
session.sql("GRANT CREATE PROCEDURE ON SCHEMA CMO_EVAL_LAB.PUBLIC TO ROLE SYSADMIN").collect()

session.sql("""
CREATE OR REPLACE PROCEDURE CMO_EVAL_LAB.PUBLIC.RUN_SCHEDULED_EVAL()
RETURNS VARCHAR
LANGUAGE SQL
AS
BEGIN
    -- Generate a unique run name with timestamp
    LET run_name VARCHAR := 'scheduled-' || TO_CHAR(CURRENT_TIMESTAMP(), 'YYYYMMDD-HH24MISS');

    -- Start the evaluation against the production alias
    CALL EXECUTE_AI_EVALUATION(
        'START',
        OBJECT_CONSTRUCT('run_name', :run_name),
        '@CMO_EVAL_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
    );

    -- Note: In production, you would poll STATUS in a loop here.
    -- For this lab, we demonstrate the score-check logic separately.

    RETURN 'Evaluation started: ' || :run_name;
END
""").collect()

print("Stored procedure RUN_SCHEDULED_EVAL created.")

In [ ]:
# Create a Snowflake Task to run evals on a schedule
# (Created in SUSPENDED state for the lab — resume it when ready for production)
session.sql("""
CREATE OR REPLACE TASK DAILY_EVAL_CHECK
    WAREHOUSE = CMO_EVAL_WH
    SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
    COMMENT = 'Daily evaluation regression check for CMO Assistant'
AS
    CALL RUN_SCHEDULED_EVAL()
""").collect()

print("Task DAILY_EVAL_CHECK created (suspended).")
print("To activate in production: ALTER TASK DAILY_EVAL_CHECK RESUME;")
session.sql("SHOW TASKS IN SCHEMA CMO_EVAL_LAB.PUBLIC").show()

In [ ]:
# Regression detection query — compare latest run to the previous run
# This would be called by a downstream task or alert after the eval completes
session.sql("""
WITH ranked_runs AS (
    SELECT
        RUN_NAME,
        METRIC_NAME,
        AVG_SCORE,
        RUN_TIMESTAMP,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME ORDER BY RUN_TIMESTAMP DESC) AS rn
    FROM EVAL_SCORE_HISTORY
),
latest AS (SELECT * FROM ranked_runs WHERE rn = 1),
previous AS (SELECT * FROM ranked_runs WHERE rn = 2)
SELECT
    l.METRIC_NAME,
    p.AVG_SCORE AS PREVIOUS_SCORE,
    l.AVG_SCORE AS LATEST_SCORE,
    ROUND(l.AVG_SCORE - p.AVG_SCORE, 4) AS DELTA,
    CASE
        WHEN l.AVG_SCORE < p.AVG_SCORE - 0.05 THEN '⚠️ REGRESSION'
        WHEN l.AVG_SCORE > p.AVG_SCORE + 0.05 THEN '✅ IMPROVEMENT'
        ELSE '— STABLE'
    END AS STATUS
FROM latest l
JOIN previous p ON l.METRIC_NAME = p.METRIC_NAME
ORDER BY DELTA ASC
""").show()

---
## Section 7: Human Feedback Integration

LLM-as-judge gives you scalable, automated scoring. But it's only as good as its calibration to human judgment.

In this section, we'll:
1. Query user feedback from the official AI Observability events table (`SNOWFLAKE.LOCAL.AI_OBSERVABILITY_EVENTS`)
2. Use `CORTEX.COMPLETE` to classify negative feedback text automatically
3. Show how to identify candidate questions for the eval dataset from production feedback

Feedback is stored as observability events with `RECORD:name = 'CORTEX_AGENT_FEEDBACK'`. See: [Monitor Cortex Agent requests](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-monitor#view-feedback-provided-by-users)

In [ ]:
# Execute requests against the agent and submit simulated human feedback
# Uses the official Feedback REST API: POST /api/v2/databases/{db}/schemas/{schema}/agents/{name}:feedback
import json
import requests as req

# Get host and token from the session connection
host = session.connection.host
with open('/snowflake/session/token', 'r') as f:
    token = f.read().strip()

feedback_url = f"https://{host}/api/v2/databases/CMO_EVAL_LAB/schemas/PUBLIC/agents/CMO_ASSISTANT:feedback"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "X-Snowflake-Authorization-Token-Type": "OAUTH"
}

# Define test interactions with expected feedback signals
test_interactions = [
    {"query": "What was total spend in 2024?", "positive": True, "message": ""},
    {"query": "Which channel had the highest ROI in Q4 2024?", "positive": True, "message": "Exactly what I needed"},
    {"query": "Compare our Q4 spend to strategy recommendations", "positive": False, "message": "Only showed spend data but didnt reference the strategy document"},
    {"query": "Give me an executive summary of H1 performance", "positive": False, "message": "The format wasnt right, just a wall of text not an executive brief"},
    {"query": "What are our competitors doing?", "positive": True, "message": "Good that it declined the out-of-scope question"},
    {"query": "Whats our attribution methodology?", "positive": False, "message": "Missed the key detail about the 30-day lookback window"},
]

print(f"Sending {len(test_interactions)} requests to the agent and submitting feedback...\n")

for i, interaction in enumerate(test_interactions):
    # Step 1: Call the agent and capture request_id
    result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'CMO_EVAL_LAB.PUBLIC.CMO_ASSISTANT',
      $${{
        "messages": [
          {{
            "role": "user",
            "content": [
              {{"type": "text", "text": "{interaction['query']}"}}
            ]
          }}
        ]
      }}$$
    ) AS response
    """).collect()

    resp = json.loads(result[0]['RESPONSE'])
    request_id = resp.get('request_id', '')

    # Step 2: Submit feedback via REST API
    feedback_body = {
        "orig_request_id": request_id,
        "positive": interaction["positive"],
    }
    if interaction["message"]:
        feedback_body["feedback_message"] = interaction["message"]

    fb_resp = req.post(feedback_url, headers=headers, json=feedback_body)
    signal = "thumbs-up" if interaction["positive"] else "thumbs-down"

    if fb_resp.status_code in (200, 201):
        status = "OK"
    else:
        status = f"ERR {fb_resp.status_code}"
        if i == 0:
            print(f"  DEBUG: {fb_resp.text[:300]}")

    print(f"  [{i+1}/{len(test_interactions)}] {interaction['query'][:50]:50s} -> {signal:11s} [{status}]")

print(f"\nDone. Submitted feedback for {len(test_interactions)} agent interactions.")
print("Events will appear in SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS after propagation.")

In [ ]:
# Query user feedback from the official AI Observability events table
# Feedback events have: VALUE:positive (bool), VALUE:feedback_message (text), VALUE:entity_type
session.sql("""
SELECT
    TIMESTAMP AS FEEDBACK_TIME,
    RECORD_ATTRIBUTES:"snow.ai.observability.user.name"::VARCHAR AS USER_NAME,
    CASE WHEN VALUE:positive::BOOLEAN THEN 'positive' ELSE 'negative' END AS FEEDBACK_SIGNAL,
    VALUE:feedback_message::VARCHAR AS FEEDBACK_MESSAGE,
    VALUE:entity_type::VARCHAR AS ENTITY_TYPE,
    RECORD_ATTRIBUTES:"snow.ai.observability.session.id"::VARCHAR AS SESSION_ID
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB',
    'PUBLIC',
    'CMO_ASSISTANT',
    'CORTEX AGENT'
))
WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
ORDER BY TIMESTAMP DESC
""").show(max_width=120)

# Summary stats
session.sql("""
SELECT
    COUNT(*) AS TOTAL_FEEDBACK,
    SUM(CASE WHEN VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) AS THUMBS_UP,
    SUM(CASE WHEN NOT VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) AS THUMBS_DOWN,
    ROUND(SUM(CASE WHEN VALUE:positive::BOOLEAN THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS APPROVAL_RATE_PCT
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB',
    'PUBLIC',
    'CMO_ASSISTANT',
    'CORTEX AGENT'
))
WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
""").show()

In [ ]:
# Use CORTEX.COMPLETE to classify negative feedback into issue categories
# This gives you an aggregate signal on WHAT is going wrong without manual review
session.sql("""
WITH negative_feedback AS (
    SELECT
        VALUE:feedback_message::VARCHAR AS FEEDBACK_MESSAGE,
        TIMESTAMP
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
        'CMO_EVAL_LAB',
        'PUBLIC',
        'CMO_ASSISTANT',
        'CORTEX AGENT'
    ))
    WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
      AND VALUE:positive::BOOLEAN = FALSE
      AND VALUE:feedback_message IS NOT NULL
      AND VALUE:feedback_message != ''
)
SELECT
    FEEDBACK_MESSAGE,
    TRIM(SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        'Classify this user feedback about an AI assistant into exactly one category. '
        || 'Categories: WRONG_ANSWER, MISSING_CONTEXT, FORMAT_ISSUE, TOOL_SELECTION, SCOPE_ERROR, OTHER. '
        || 'Return only the category name, nothing else. '
        || 'Feedback: ' || FEEDBACK_MESSAGE
    )) AS ISSUE_CATEGORY
FROM negative_feedback
ORDER BY TIMESTAMP DESC
""").show(max_width=120)

In [ ]:
# Active sampling: identify negative-feedback sessions to investigate
# These represent real failures that should become eval dataset candidates
session.sql("""
SELECT
    VALUE:feedback_message::VARCHAR AS FAILURE_SIGNAL,
    RECORD_ATTRIBUTES:"snow.ai.observability.session.id"::VARCHAR AS SESSION_ID,
    TIMESTAMP AS FEEDBACK_TIME,
    '-- TODO: Retrieve the original query from the session request events --' AS NEXT_STEP
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_OBSERVABILITY_EVENTS(
    'CMO_EVAL_LAB',
    'PUBLIC',
    'CMO_ASSISTANT',
    'CORTEX AGENT'
))
WHERE RECORD:name = 'CORTEX_AGENT_FEEDBACK'
  AND VALUE:positive::BOOLEAN = FALSE
  AND VALUE:feedback_message IS NOT NULL
ORDER BY TIMESTAMP DESC
""").show(max_width=120)

print("""
Next steps for these candidates:
1. Use the SESSION_ID to join with request events and retrieve the original user query
2. Review the failure signal to understand what went wrong
3. Write ground truth describing what a CORRECT response looks like
4. Add to your eval dataset and re-run evaluation
""")

---
## Section 8: CI/CD Quality Gates

In production, evaluations run automatically as part of your deployment pipeline. The recommended pattern uses **dbt seeds** for the eval dataset (version-controlled in git) and **GitHub Actions** with the Snowflake CLI to trigger evals on every PR.

This section shows the SQL components that a CI/CD pipeline would call. The pipeline itself runs in GitHub Actions using the `snowflakedb/snowflake-cli-action@v2.0` with OIDC authentication.

### Architecture:
```
PR opened → deploy staging agent → dbt seed (load eval dataset) →
EXECUTE_AI_EVALUATION → check scores → pass/fail the PR
```

In [ ]:
# CI/CD threshold gate — this query would be called from a GitHub Actions step
# to determine whether a PR passes or fails the eval quality gate
#
# In your GitHub Action:
#   snow sql -q "SELECT * FROM TABLE(...)" -x | check_threshold.py

session.sql("""
WITH run_scores AS (
    SELECT
        METRIC_NAME,
        ROUND(AVG(EVAL_AGG_SCORE), 4) AS AVG_SCORE,
        COUNT(*) AS NUM_QUESTIONS
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'CMO_EVAL_LAB', 'PUBLIC', 'CMO_ASSISTANT', 'CORTEX AGENT', 'improved-v2'
    ))
    GROUP BY METRIC_NAME
)
SELECT
    METRIC_NAME,
    AVG_SCORE,
    NUM_QUESTIONS,
    CASE
        WHEN METRIC_NAME = 'answer_correctness' AND AVG_SCORE < 0.70 THEN 'FAIL'
        WHEN METRIC_NAME = 'logical_consistency' AND AVG_SCORE < 0.60 THEN 'FAIL'
        WHEN METRIC_NAME = 'tool_selection' AND AVG_SCORE < 0.65 THEN 'FAIL'
        ELSE 'PASS'
    END AS GATE_STATUS,
    CASE
        WHEN METRIC_NAME = 'answer_correctness' THEN 0.70
        WHEN METRIC_NAME = 'logical_consistency' THEN 0.60
        WHEN METRIC_NAME = 'tool_selection' THEN 0.65
    END AS THRESHOLD
FROM run_scores
ORDER BY GATE_STATUS DESC, AVG_SCORE ASC
""").show()

In [ ]:
# Regression attribution — when a scheduled eval detects a score drop,
# correlate it with the agent version history to identify the culprit
session.sql("""
SHOW VERSIONS IN AGENT CMO_ASSISTANT
""").show()

print("""
Regression attribution workflow:
1. Scheduled eval detects score drop (Section 6 query)
2. Check SHOW VERSIONS IN AGENT to find versions created since last passing run
3. Cross-reference version COMMENT and timestamp with git history
4. Identify which change (semantic view update, instruction edit, tool config) caused the drop
5. Fix and re-evaluate before promoting
""")

In [ ]:
# Example: What a dbt seed file for eval questions looks like
# File: seeds/eval_questions.csv in your dbt project repo
#
# This CSV is version-controlled in git. Any team member can add questions via PR.
# On merge, `dbt seed` loads it into the EVAL_QUESTIONS table in Snowflake.

print("""
# seeds/eval_questions.csv
# ─────────────────────────────────────────────────────────────────
# input_query,ground_truth
# "What was total spend in 2024?","{\"ground_truth_output\": \"Total spend was approximately $1.73M...\"}"
# "Which channel had highest ROI in Q4?","{\"ground_truth_output\": \"Email had highest ROI at ~11.3x...\"}"
# "What is our attribution methodology?","{\"ground_truth_output\": \"Multi-touch attribution with 30-day lookback...\"}"
# ...
#
# GitHub Actions workflow (pr_eval_gate.yml):
# ─────────────────────────────────────────────────────────────────
# steps:
#   - uses: snowflakedb/snowflake-cli-action@v2.0
#     with:
#       use-oidc: true
#
#   - name: Load eval dataset
#     run: snow dbt execute -x my_dbt_project seed --select eval_questions
#
#   - name: Run evaluation
#     run: |
#       snow sql -q "CALL EXECUTE_AI_EVALUATION('START', \
#         OBJECT_CONSTRUCT('run_name', 'pr-${{ github.event.number }}'), \
#         '@EVAL_STAGE/eval_config.yaml')" -x
#
#   - name: Check threshold
#     run: |
#       RESULT=$(snow sql -q "SELECT CASE WHEN MIN(GATE_STATUS) = 'PASS' THEN 'PASS' ELSE 'FAIL' END \
#         FROM (...threshold query...)" -x --format json)
#       if [[ "$RESULT" == *"FAIL"* ]]; then exit 1; fi
""")

---
## Summary: What You Accomplished

In this lab you completed the full Cortex Agent evaluation lifecycle **and** set up the automation layer for continuous quality:

| Step | What you did |
|------|-------------|
| **Build** | Created a multi-tool agent (Analyst + Search + Skill) |
| **Dataset** | Designed 12 eval questions covering happy paths, multi-tool, edge cases, and refusals |
| **Metrics** | Used 2 built-in metrics + 1 custom `tool_selection` metric |
| **Evaluate** | Ran a baseline evaluation and inspected per-question scores and explanations |
| **Iterate** | Identified weak areas and improved orchestration instructions |
| **Compare** | Ran a second evaluation and compared scores side-by-side |
| **Promote** | Committed the improved version and assigned the `production` alias |
| **Automate** | Set up scheduled regression detection with Tasks and score history |
| **Feedback** | Built a human feedback pipeline with CORTEX.COMPLETE classification |
| **CI/CD** | Designed threshold gates for automated PR quality checks |

### What's Next?

- **Activate the Task** — `ALTER TASK DAILY_EVAL_CHECK RESUME` to start scheduled regression detection
- **Connect real feedback** — Wire your application's thumbs-up/thumbs-down UI to the USER_FEEDBACK table
- **Set up notifications** — Add a notification integration to alert on regression (Slack, email, PagerDuty)
- **Build the dbt seed** — Move EVAL_QUESTIONS to a `seeds/eval_questions.csv` in your dbt project for git-based collaboration
- **Implement the GitHub Action** — Use the `snowflakedb/snowflake-cli-action@v2.0` with the threshold gate query to block PRs that degrade quality


In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS CMO_EVAL_LAB CASCADE").collect()
# print("Lab resources cleaned up.")